# Zadanie 5: własny serwer MCP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adrianstando/2026Z-PJATK-TEG/blob/main/05-mcp-multi-agent/zadanie.ipynb)

**Oddanie:** mail z tematem `[TEG] lab-05`, w załączniku ten notebook **z wynikami komórek** i plik serwera `biblioteka_mcp.py`, bez kluczy API.

**Termin:** przed zajęciami 6.

Napisz serwer MCP dla domowej biblioteki książek: kilka narzędzi nad małym plikiem JSON. Potem sprawdź go bez modelu i podłącz do agenta.

Części:
1. Serwer `biblioteka_mcp.py` w `FastMCP` z co najmniej pięcioma narzędziami.
2. Bezpośrednie wywołanie serwera MCP, bez modelu.
3. Podłączenie serwera MCP do agenta: LangChain albo agent kodujący (Claude Code, Codex, VS Code, Claude Desktop).

Kod z [notebooka z zajęć](notebook.ipynb) można kopiować i modyfikować.

## 0. Konfiguracja

In [ ]:
# Colab / Deepnote / Codespaces: odkomentuj i uruchom raz
# %pip install -q openai python-dotenv "mcp<2" langchain langchain-openai langgraph langchain-mcp-adapters

Dostawcę modelu wybiera się zmienną `LLM_PROVIDER`, tak jak w notebooku z zajęć ([00-dostep-do-llm](../00-dostep-do-llm/README.md)). Model jest potrzebny tylko w wariancie A części 3.

In [ ]:
import os

try:  # Colab: sekrety z panelu "Secrets"
    from google.colab import userdata
    for k in ["LLM_PROVIDER", "OLLAMA_CHAT", "OPENAI_API_KEY", "AZURE_OPENAI_ENDPOINT",
              "AZURE_OPENAI_API_KEY", "DEEPSEEK_API_KEY"]:
        try:
            os.environ.setdefault(k, userdata.get(k))
        except Exception:
            pass
except ImportError:  # lokalnie: plik .env
    from dotenv import load_dotenv
    load_dotenv(override=True)

from openai import OpenAI, AzureOpenAI

MODELS = {
    "openai":   {"chat": "gpt-5-nano", "classic": "gpt-4.1-mini"},
    "azure":    {"chat": "gpt-4o-mini", "classic": "gpt-4o-mini"},
    "deepseek": {"chat": "deepseek-chat", "classic": "deepseek-chat"},
    "ollama":   {"chat": os.getenv("OLLAMA_CHAT", "gemma4:e4b"), "classic": "gemma4:e4b"},
    "proxy":    {"chat": "copilot/gpt-5-mini", "classic": "copilot/gpt-5-mini"},
}

def make_client(provider):
    if provider == "azure":
        return AzureOpenAI(api_version="2024-12-01-preview")
    if provider == "deepseek":
        return OpenAI(base_url="https://api.deepseek.com", api_key=os.environ["DEEPSEEK_API_KEY"])
    if provider == "ollama":
        return OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    if provider == "proxy":
        return OpenAI(base_url="http://localhost:8787/v1", api_key="teg")
    return OpenAI()

PROVIDER = os.getenv("LLM_PROVIDER", "ollama")
client = make_client(PROVIDER)
CHAT, CLASSIC = MODELS[PROVIDER]["chat"], MODELS[PROVIDER]["classic"]
AGENT = CLASSIC  # model do pętli agenta
print(f"czat: {PROVIDER} ({CHAT}, klasyczny: {CLASSIC}) | embeddingi: nieużywane")

In [ ]:
from langchain_openai import ChatOpenAI, AzureChatOpenAI

def make_llm(model):
    if PROVIDER == "azure":
        return AzureChatOpenAI(azure_deployment=model, api_version="2024-12-01-preview")
    return ChatOpenAI(model=model, base_url=str(client.base_url), api_key=client.api_key)

llm = make_llm(AGENT)

## Część 1. Serwer

Komórka poniżej zapisuje dane biblioteki do pliku `biblioteka.json`: dwadzieścia książek z polami `id`, `title`, `author`, `year`, `genre`, `status` (`"przeczytana"`, `"w trakcie"`, `"do przeczytania"`) i `rating` (1–5 albo `null`).

In [ ]:
import json

BOOKS = [
    {"id": 1, "title": "Solaris", "author": "Stanisław Lem", "year": 1961, "genre": "science fiction", "status": "przeczytana", "rating": 5},
    {"id": 2, "title": "Cyberiada", "author": "Stanisław Lem", "year": 1965, "genre": "science fiction", "status": "do przeczytania", "rating": None},
    {"id": 3, "title": "Niezwyciężony", "author": "Stanisław Lem", "year": 1964, "genre": "science fiction", "status": "do przeczytania", "rating": None},
    {"id": 4, "title": "Lód", "author": "Jacek Dukaj", "year": 2007, "genre": "science fiction", "status": "do przeczytania", "rating": None},
    {"id": 5, "title": "Inne pieśni", "author": "Jacek Dukaj", "year": 2003, "genre": "science fiction", "status": "przeczytana", "rating": 4},
    {"id": 6, "title": "Lalka", "author": "Bolesław Prus", "year": 1890, "genre": "powieść", "status": "przeczytana", "rating": 4},
    {"id": 7, "title": "Quo vadis", "author": "Henryk Sienkiewicz", "year": 1896, "genre": "powieść historyczna", "status": "przeczytana", "rating": 3},
    {"id": 8, "title": "Pan Tadeusz", "author": "Adam Mickiewicz", "year": 1834, "genre": "epopeja", "status": "przeczytana", "rating": 4},
    {"id": 9, "title": "Bieguni", "author": "Olga Tokarczuk", "year": 2007, "genre": "powieść", "status": "w trakcie", "rating": None},
    {"id": 10, "title": "Księgi Jakubowe", "author": "Olga Tokarczuk", "year": 2014, "genre": "powieść historyczna", "status": "do przeczytania", "rating": None},
    {"id": 11, "title": "Empuzjon", "author": "Olga Tokarczuk", "year": 2022, "genre": "powieść", "status": "do przeczytania", "rating": None},
    {"id": 12, "title": "Rdza", "author": "Jakub Małecki", "year": 2017, "genre": "powieść", "status": "przeczytana", "rating": 5},
    {"id": 13, "title": "Chłopki. Opowieść o naszych babkach", "author": "Joanna Kuciel-Frydryszak", "year": 2023, "genre": "reportaż", "status": "w trakcie", "rating": None},
    {"id": 14, "title": "Kasacja", "author": "Remigiusz Mróz", "year": 2015, "genre": "kryminał", "status": "przeczytana", "rating": 4},
    {"id": 15, "title": "Zaginięcie", "author": "Remigiusz Mróz", "year": 2016, "genre": "kryminał", "status": "do przeczytania", "rating": None},
    {"id": 16, "title": "Wiedźmin. Ostatnie życzenie", "author": "Andrzej Sapkowski", "year": 1993, "genre": "fantasy", "status": "przeczytana", "rating": 5},
    {"id": 17, "title": "Gra o tron", "author": "George R.R. Martin", "year": 1996, "genre": "fantasy", "status": "do przeczytania", "rating": None},
    {"id": 18, "title": "Harry Potter i Kamień Filozoficzny", "author": "J.K. Rowling", "year": 1997, "genre": "fantasy", "status": "przeczytana", "rating": 4},
    {"id": 19, "title": "Zwiadowcy. Ruiny Gorlanu", "author": "John Flanagan", "year": 2004, "genre": "fantasy", "status": "do przeczytania", "rating": None},
    {"id": 20, "title": "Rok 1984", "author": "George Orwell", "year": 1949, "genre": "dystopia", "status": "przeczytana", "rating": 5},
]
with open("biblioteka.json", "w", encoding="utf-8") as f:
    json.dump(BOOKS, f, ensure_ascii=False, indent=1)
print(len(BOOKS), "książek w biblioteka.json")

Napisz plik `biblioteka_mcp.py` (w komórce `%%writefile` poniżej albo jako osobny plik) z serwerem `FastMCP("biblioteka")`, który czyta i zapisuje `biblioteka.json`.

1. Napisz co najmniej **pięć narzędzi**: co najmniej trzy tylko do odczytu i co najmniej jedno, które zmienia dane. Przykłady: wyszukiwanie po tytule lub autorze, lista książek o danym statusie, szczegóły książki, statystyki (ile przeczytanych, średnia ocena), dodanie książki, zmiana statusu i oceny.
2. Upewnij się, że każde narzędzie ma typy argumentów i docstring: z nich powstają schemat i opis w `tools/list`.
3. Dodaj adnotacje `ToolAnnotations`: `readOnlyHint=True` przy narzędziach do odczytu, a przy narzędziach, które zmieniają dane, `readOnlyHint=False` i odpowiedni `destructiveHint`.
4. Sprawdź, czy złe argumenty (np. nieistniejące `id`, ocena spoza 1–5) dają czytelny komunikat błędu, a nie wyjątek bez opisu.
5. Uruchom serwer przez stdio: `mcp.run("stdio")` w bloku `if __name__ == "__main__":`.

In [ ]:
%%writefile biblioteka_mcp.py
"""Serwer MCP domowej biblioteki (dane w biblioteka.json)."""

import json

from mcp.server.fastmcp import FastMCP
from mcp.types import ToolAnnotations

mcp = FastMCP("biblioteka", log_level="WARNING")

# TODO: odczyt i zapis biblioteka.json, co najmniej pięć narzędzi z adnotacjami


if __name__ == "__main__":
    mcp.run("stdio")

Serwera stdio nie uruchamia się ręcznie w tle: proces serwera startuje klient MCP (część 2) albo aplikacja (część 3) i rozmawia z nim przez stdin/stdout. Żeby szybko sprawdzić, czy plik się wczytuje i jakie narzędzia widzi FastMCP, wystarczy go zaimportować w osobnym procesie (`!` uruchamia polecenie powłoki):

In [ ]:
!python -c "import asyncio, biblioteka_mcp as b; print([t.name for t in asyncio.run(b.mcp.list_tools())])"

## Część 2. Bezpośrednie wywołanie serwera MCP

Napisz klienta MCP z SDK (`ClientSession`), który bez modelu łączy się z serwerem przez stdio i:

1. wypisuje nazwy narzędzi z `tools/list` razem z adnotacją `readOnlyHint`,
2. wywołuje jedno narzędzie do odczytu z argumentami i wypisuje wynik,
3. wywołuje narzędzie z błędnym argumentem i wypisuje, co wróciło (`isError` i treść).

In [ ]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

BIBLIOTEKA = StdioServerParameters(command=sys.executable, args=["biblioteka_mcp.py"])

# TODO: sesja, tools/list z readOnlyHint, wywołanie narzędzia, wywołanie z błędnym argumentem

## Część 3. Podłączenie serwera MCP do agenta

Wybierz jeden wariant.

**Wariant A: agent w LangChain.** Załaduj narzędzia serwera przez `MultiServerMCPClient` i `load_mcp_tools` z `langchain-mcp-adapters`, a potem zbuduj agenta `create_agent`. Zadaj mu dwa pytania i przy każdym wypisz wywołania narzędzi i odpowiedź:
- „Które książki Lema mam jeszcze do przeczytania?”
- pytanie, po którym agent zmienia dane, np. „Skończyłem Biegunów, daj im ocenę 4 i zapisz, że zaczynam Lód”; po odpowiedzi odczytaj plik `biblioteka.json` i pokaż zmianę.

**Wariant B: agent kodujący.** Dodaj serwer do konfiguracji Claude Code, Codexa, VS Code albo Claude Desktop (konfiguracje i polecenia są na slajdzie „Podłączenie serwera”) i zadaj w aplikacji te same dwa pytania. W komórce poniżej wklej użytą konfigurację (plik JSON albo polecenie).

**Uwaga: w wariancie B dołącz do maila zrzuty ekranu** z listą narzędzi serwera w aplikacji (np. `/mcp` w Claude Code) i z obiema rozmowami, na których widać wywołania narzędzi.

In [ ]:
# TODO: wariant A (agent i dwa pytania) albo wariant B (konfiguracja aplikacji jako tekst)